In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm.notebook import tqdm
from scipy import stats

# 1. Данные из задания про страты

Будем проверять гипотезу о равенстве средних метрики `y`.

In [ ]:
df = pd.read_csv('./04-strat-hw/04-strat-hw-data-public.csv')
df.head()

Добавим значение метрики за прошлый период `y_before`, которая скоррелирована с `y`.

In [ ]:
mean_ = df['y'].mean()
std_ = df['y'].std()
len_ = len(df)
y_before = np.random.normal(mean_, std_, len_)
df['y_before'] = y_before * 0.5 + df['y'].values * 0.5

Убедимся что есть корреляция

In [ ]:
df.corrwith(df['y'])

# 2. Оценка разницы средних для приращений

Даёт близкий к CUPED результат, если есть сильная положительная корреляция.

In [ ]:
def plot_pvalue_distribution_power(dict_pvalues, alpha=0.05):
    """Рисует графики распределения pvalue."""
    X = np.linspace(0, 1, 1000)
    for key, pvalues in dict_pvalues.items():
        Y = [np.mean(pvalues <= x) for x in X]
        prob_p = np.mean(np.array(pvalues) < alpha)
        plt.plot(X, Y, label=f'{key}, prob_p={prob_p:0.2f}')
    plt.plot([alpha, alpha], [0, 1], '--k', alpha=0.8)
    plt.plot([0, 1], [0, 1], '--k', alpha=0.8)
    plt.title('Оценка распределения p-value', size=16)
    plt.xlabel('p-value', size=12)
    plt.legend(fontsize=12)
    plt.grid()
    plt.show()
    
def check_ttest(df_a, df_b, metric_name, cov_name):
    """Проверяет гипотезу о равенстве средних с помощью t-test.

    return - pvalue.
    """
    values_a = df_a[metric_name].values
    values_b = df_b[metric_name].values
    _, pvalue = stats.ttest_ind(values_a, values_b)
    return pvalue

def check_diff_ttest(df_a, df_b, metric_name, cov_name):
    """Проверяет гипотезу о равенстве средних с помощью t-test.

    return - pvalue.
    """
    values_a = df_a[metric_name].values - df_a[cov_name].values
    values_b = df_b[metric_name].values - df_b[cov_name].values
    _, pvalue = stats.ttest_ind(values_a, values_b)
    return pvalue

In [ ]:
n_iter = 3000
alpha = 0.05
size = 100
effect = 40

dict_pvalues = defaultdict(list)
for _ in tqdm(range(n_iter)):
    indexes_a, indexes_b = np.random.choice(np.arange(len_), (2, size), False)
    df_a = df.iloc[indexes_a].copy()
    df_b = df.iloc[indexes_b].copy()
    dict_pvalues['ttest AA'].append(check_ttest(df_a, df_b, 'y', 'y_before'))
    dict_pvalues['ttest diff AA'].append(check_diff_ttest(df_a, df_b, 'y', 'y_before'))
    df_b['y'] += effect
    dict_pvalues['ttest AB'].append(check_ttest(df_a, df_b, 'y', 'y_before'))
    dict_pvalues['ttest diff AB'].append(check_diff_ttest(df_a, df_b, 'y', 'y_before'))

In [ ]:
plot_pvalue_distribution_power(dict_pvalues)

# 3. CUPED

Напишем функции для применения CUPED

In [ ]:
def calculate_theta(metrics, covariates):
    """Вычисляем Theta.

    metrics - значения исходной метрики
    covariates - значения ковариаты

    return - theta.
    """
    covariance = np.cov(covariates, metrics)[0, 1]
    variance = covariates.var()
    theta = covariance / variance
    return theta

def check_cuped(df_a, df_b, df_theta, metric_name, cov_name):
    """Проверяет гипотезу о равенстве средних с использованием CUPED.
    
    df_control и df_pilot - данные контрольной и экспериментальной групп
    df_theta - данные для оценки theta

    return - pvalue.
    """
    theta = calculate_theta(df_theta[metric_name], df_theta[cov_name])
    metric_cuped_a = df_a[metric_name] - theta * df_a[cov_name]
    metric_cuped_b = df_b[metric_name] - theta * df_b[cov_name]
    _, pvalue = stats.ttest_ind(metric_cuped_a, metric_cuped_b)
    return pvalue

In [ ]:
n_iter = 3000
alpha = 0.05
size = 100
effect = 40

dict_pvalues = defaultdict(list)
for _ in tqdm(range(n_iter)):
    indexes_a, indexes_b = np.random.choice(np.arange(len_), (2, size), False)
    df_a = df.iloc[indexes_a].copy()
    df_b = df.iloc[indexes_b].copy()
    df_theta = pd.concat([df_a, df_b])
    dict_pvalues['ttest AA'].append(check_ttest(df_a, df_b, 'y', 'y_before'))
    dict_pvalues['cuped AA'].append(check_cuped(df_a, df_b, df_theta, 'y', 'y_before'))
    df_b['y'] += effect
    df_theta = pd.concat([df_a, df_b])
    dict_pvalues['ttest AB'].append(check_ttest(df_a, df_b, 'y', 'y_before'))
    dict_pvalues['cuped AB'].append(check_cuped(df_a, df_b, df_theta, 'y', 'y_before'))
    dict_pvalues['ttest diff AB'].append(check_diff_ttest(df_a, df_b, 'y', 'y_before'))

In [ ]:
plot_pvalue_distribution_power(dict_pvalues)

### Где можно ошибиться

#### 1. Центрировать разными константами

```
    metric_cuped_a = df_a['metric'] - theta * (df_a['covariate'] - df_a['covariate'].mean())
    metric_cuped_b = df_b['metric'] - theta * (df_b['covariate'] - df_b['covariate'].mean())
```

#### 2. Использовать разные Theta

```
    metric_cuped_a = df_a['metric'] - calculate_theta(df_a['metric'], df_a['covariate']) * df_a['covariate']
    metric_cuped_b = df_b['metric'] - calculate_theta(df_b['metric'], df_b['covariate']) * df_b['covariate']
```

#### 3. Вычислять Theta по одной группе, когда размеры групп малы

```
    sample_size = 5
    theta = calculate_theta(df_a['metric'], df_a['covariate'])
    metric_cuped_a = df_a['metric'] - theta * df_a['covariate']
    metric_cuped_b = df_b['metric'] - theta * df_b['covariate']
```

### На чём считать Theta ?

- Если размеры групп достаточно велики, лучше оценивать Theta на объединённых данных контрольной и экспериментальной групп;
- Если размеры групп малы, то оценка Theta на исторических данных может дать лучшее качество.

# 4. CUPED + ML

Можно создавать ковариату с большей корреляцией на основе ML прогноза целевой переменной.

In [ ]:
from lightgbm import LGBMRegressor

In [ ]:
df_train = df.iloc[:len_ // 2].copy()
df_test = df.iloc[len_ // 2:].copy()

In [ ]:
feature_names = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'y_before']
model = LGBMRegressor().fit(df_train[feature_names], df_train['y'])
df_test['y_pred'] = model.predict(df_test[feature_names])

In [ ]:
df_test.corrwith(df_test['y'])

In [ ]:
n_iter = 3000
alpha = 0.05
size = 100
effect = 60

dict_pvalues = defaultdict(list)
for _ in tqdm(range(n_iter)):
    indexes_a, indexes_b = np.random.choice(np.arange(len(df_test)), (2, size), False)
    df_a = df_test.iloc[indexes_a].copy()
    df_b = df_test.iloc[indexes_b].copy()
    df_theta = pd.concat([df_a, df_b])
    dict_pvalues['cuped AA'].append(check_cuped(df_a, df_b, df_theta, 'y', 'y_before'))
    dict_pvalues['cuped ML AA'].append(check_cuped(df_a, df_b, df_theta, 'y', 'y_pred'))
    df_b['y'] += effect
    df_theta = pd.concat([df_a, df_b])
    dict_pvalues['cuped AB'].append(check_cuped(df_a, df_b, df_theta, 'y', 'y_before'))
    dict_pvalues['cuped ML AB'].append(check_cuped(df_a, df_b, df_theta, 'y', 'y_pred'))

In [ ]:
plot_pvalue_distribution_power(dict_pvalues)

# 5. CUPED vs Stratification

https://habr.com/ru/companies/X5Tech/articles/826488/